In [ ]:
# Colab package install
!pip -q install "numpy<2" scipy pandas matplotlib pillow opencv-python-headless==4.5.5.64 insightface onnxruntime
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!rm -rf /content/ai-hub-models
!git clone --branch v0.49.1 --depth 1 https://github.com/quic/ai-hub-models.git /content/ai-hub-models
!pip -q install -e "/content/ai-hub-models[facemap-3dmm]"

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("tensorflow") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow==2.17.1"])

print("install complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 18.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 90.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14

In [ ]:
!pip -q install "numpy<2" scipy pandas matplotlib pillow opencv-python-headless==4.5.5.64 insightface onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.5/41.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.6/123.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 11.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albucore 0.0.24 requires opencv-python-headless>=4.9.0.80, but you have opencv-python-headless 4.5.5.64 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have 

In [ ]:
import io
import json
import math
import os
import sys
import urllib.request
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import files
from insightface.app import FaceAnalysis

sys.path.insert(0, "/content/ai-hub-models")

from qai_hub_models.models.facemap_3dmm.utils import CachedWebModelAsset, load_numpy

MODEL_INPUT_SIZE = 128
MODEL_ID = "facemap_3dmm"
MODEL_ASSET_VERSION = 1
RUNTIME_DIR = Path("/content/facemap_3dmm_runtime")
MODEL_DIR = RUNTIME_DIR / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Qualcomm official hosted model file on Hugging Face.
# The repo page currently points to the v0.49.1 release family.
TFLITE_URLS = [
    "https://huggingface.co/qualcomm/Facial-Landmark-Detection/resolve/main/Facial-Landmark-Detection_w8a8.tflite",
    "https://huggingface.co/qualcomm/Facial-Landmark-Detection/resolve/main/Facial-Landmark-Detection_w8a8.tflite?download=1",
]
TFLITE_MODEL_PATH = MODEL_DIR / "Facial-Landmark-Detection_w8a8.tflite"


def download_first_available(urls, dst_path):
    dst_path = Path(dst_path)
    if dst_path.exists():
        return dst_path

    last_error = None
    for url in urls:
        try:
            print(f"Downloading: {url}")
            urllib.request.urlretrieve(url, dst_path)
            print(f"Saved model to: {dst_path}")
            return dst_path
        except Exception as exc:
            last_error = exc
            print(f"Failed: {url} -> {exc}")

    raise RuntimeError(f"Could not download TFLite model. Last error: {last_error}")


def build_tflite_interpreter(model_path):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    return interpreter


def get_largest_face(faces):
    if not faces:
        return None
    return max(
        faces,
        key=lambda face: float((face.bbox[2] - face.bbox[0]) * (face.bbox[3] - face.bbox[1])),
    )


def make_square_crop_bbox(image_shape, det_bbox_xyxy, pad_ratio=0.30):
    h, w = image_shape[:2]
    x0, y0, x1, y1 = map(float, det_bbox_xyxy)
    bw = x1 - x0
    bh = y1 - y0
    cx = 0.5 * (x0 + x1)
    cy = 0.5 * (y0 + y1)
    side = max(bw, bh) * (1.0 + float(pad_ratio))
    half = side * 0.5

    crop_x0 = max(0, int(round(cx - half)))
    crop_y0 = max(0, int(round(cy - half)))
    crop_x1 = min(w, int(round(cx + half)))
    crop_y1 = min(h, int(round(cy + half)))

    if crop_x1 <= crop_x0 or crop_y1 <= crop_y0:
        raise ValueError("Invalid crop bbox after square expansion.")

    return np.asarray([crop_x0, crop_y0, crop_x1, crop_y1], dtype=np.int32)


def crop_and_resize_face(image_bgr, crop_bbox_xyxy, output_size=MODEL_INPUT_SIZE):
    x0, y0, x1, y1 = map(int, crop_bbox_xyxy)
    crop_bgr = image_bgr[y0:y1, x0:x1]
    if crop_bgr.size == 0:
        raise ValueError("Empty crop produced from bbox.")
    resized_bgr = cv2.resize(crop_bgr, (output_size, output_size), interpolation=cv2.INTER_LINEAR)
    return crop_bgr, resized_bgr


def prepare_tflite_input(face_bgr_128, input_detail):
    rgb = cv2.cvtColor(face_bgr_128, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    input_shape = tuple(int(x) for x in input_detail["shape"])
    input_dtype = np.dtype(input_detail["dtype"])

    if len(input_shape) != 4:
        raise ValueError(f"Unexpected TFLite input shape: {input_shape}")

    if input_shape[1:3] == (MODEL_INPUT_SIZE, MODEL_INPUT_SIZE):
        tensor = np.expand_dims(rgb, axis=0)
    elif input_shape[2:4] == (MODEL_INPUT_SIZE, MODEL_INPUT_SIZE):
        tensor = np.expand_dims(np.transpose(rgb, (2, 0, 1)), axis=0)
    else:
        raise ValueError(f"Unexpected input layout: {input_shape}")

    scale, zero_point = input_detail.get("quantization", (0.0, 0))
    if scale not in (None, 0.0):
        info = np.iinfo(input_dtype) if np.issubdtype(input_dtype, np.integer) else None
        tensor = np.round(tensor / scale + zero_point)
        if info is not None:
            tensor = np.clip(tensor, info.min, info.max)
        tensor = tensor.astype(input_dtype)
    else:
        tensor = tensor.astype(input_dtype)
    return tensor


def run_facemap_tflite(interpreter, face_bgr_128):
    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]

    input_tensor = prepare_tflite_input(face_bgr_128, input_detail)
    interpreter.set_tensor(input_detail["index"], input_tensor)
    interpreter.invoke()

    raw_output = interpreter.get_tensor(output_detail["index"])[0]
    raw_output = np.asarray(raw_output)
    scale, zero_point = output_detail.get("quantization", (0.0, 0))
    if scale not in (None, 0.0):
        raw_output = (raw_output.astype(np.float32) - zero_point) * scale
    else:
        raw_output = raw_output.astype(np.float32)
    return raw_output.reshape(-1)


def split_facemap_output(raw_output):
    raw_output = np.asarray(raw_output, dtype=np.float32).reshape(-1)
    used_output = raw_output[:264]
    extra_tail = raw_output[264:].copy()
    coeffs = split_qualcomm_3dmm_output(used_output)
    return {
        "raw_output": raw_output,
        "raw_dim": int(raw_output.size),
        "used_output": used_output,
        "used_dim": int(used_output.size),
        "unused_tail": extra_tail,
        "identity": coeffs.identity,
        "expression": coeffs.expression,
        "pose": np.asarray([coeffs.pitch, coeffs.yaw, coeffs.roll], dtype=np.float32),
        "translation_focal": np.asarray(
            [coeffs.translation_x, coeffs.translation_y, coeffs.focal_length],
            dtype=np.float32,
        ),
        "coeffs": coeffs,
    }


def draw_detection_overlay(image_bgr, det_bbox_xyxy, crop_bbox_xyxy):
    vis = image_bgr.copy()
    dx0, dy0, dx1, dy1 = map(int, det_bbox_xyxy)
    cx0, cy0, cx1, cy1 = map(int, crop_bbox_xyxy)
    cv2.rectangle(vis, (dx0, dy0), (dx1, dy1), (0, 255, 0), 2)
    cv2.putText(vis, "InsightFace det bbox", (dx0, max(24, dy0 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    cv2.rectangle(vis, (cx0, cy0), (cx1, cy1), (255, 165, 0), 2)
    cv2.putText(vis, "facemap crop bbox", (cx0, min(vis.shape[0] - 8, cy1 + 22)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 165, 0), 2)
    return vis


def draw_landmarks_on_crop(crop_bgr_128, landmarks_crop_xy):
    vis = crop_bgr_128.copy()
    for idx, (x, y) in enumerate(np.asarray(landmarks_crop_xy, dtype=np.float32)):
        cv2.circle(vis, (int(round(x)), int(round(y))), 1, (0, 255, 255), -1)
        cv2.putText(vis, str(idx), (int(round(x)) + 1, int(round(y)) - 1), cv2.FONT_HERSHEY_SIMPLEX, 0.25, (255, 255, 255), 1)
    return vis


def draw_landmarks_on_image(image_bgr, landmarks_image_xy):
    vis = image_bgr.copy()
    for idx, (x, y) in enumerate(np.asarray(landmarks_image_xy, dtype=np.float32)):
        cv2.circle(vis, (int(round(x)), int(round(y))), 2, (0, 0, 255), -1)
        cv2.putText(vis, str(idx), (int(round(x)) + 2, int(round(y)) - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 255, 255), 1)
    return vis


def show_rgb(rgb_image, title, figsize=(6, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(rgb_image)
    plt.title(title)
    plt.axis("off")
    plt.show()


def render_blendshape_bar_chart(blendshapes_dict):
    items = sorted(blendshapes_dict.items(), key=lambda kv: kv[1], reverse=True)
    names = [name for name, _ in items]
    values = [float(value) for _, value in items]
    highlight = {"jawOpen", "eyeBlinkLeft", "eyeBlinkRight", "mouthSmileLeft", "mouthSmileRight", "browInnerUp"}
    colors = ["#d95f02" if name in highlight else "#1b9e77" for name in names]

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(names, values, color=colors)
    ax.set_xlim(0.0, 1.0)
    ax.set_xlabel("normalized value")
    ax.set_title("ARKit-style 52 blendshapes from Qualcomm facemap_3dmm")
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    plt.show()
    return fig


def figure_to_bgr(fig):
    fig.canvas.draw()
    width, height = fig.canvas.get_width_height()
    rgb = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8).reshape(height, width, 3)
    plt.close(fig)
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)


def upload_one_image_bgr():
    print("Upload exactly one image file.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Please upload exactly one file.")
    file_name, data = next(iter(uploaded.items()))
    image_bgr = cv2.imdecode(np.frombuffer(data, dtype=np.uint8), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise ValueError(f"Failed to decode uploaded image: {file_name}")
    return file_name, image_bgr


from dataclasses import dataclass
from typing import Mapping, Sequence

import numpy as np


ARKIT_52_BLENDSHAPES = (
    "browDownLeft",
    "browDownRight",
    "browInnerUp",
    "browOuterUpLeft",
    "browOuterUpRight",
    "cheekPuff",
    "cheekSquintLeft",
    "cheekSquintRight",
    "eyeBlinkLeft",
    "eyeBlinkRight",
    "eyeLookDownLeft",
    "eyeLookDownRight",
    "eyeLookInLeft",
    "eyeLookInRight",
    "eyeLookOutLeft",
    "eyeLookOutRight",
    "eyeLookUpLeft",
    "eyeLookUpRight",
    "eyeSquintLeft",
    "eyeSquintRight",
    "eyeWideLeft",
    "eyeWideRight",
    "jawForward",
    "jawLeft",
    "jawOpen",
    "jawRight",
    "mouthClose",
    "mouthDimpleLeft",
    "mouthDimpleRight",
    "mouthFrownLeft",
    "mouthFrownRight",
    "mouthFunnel",
    "mouthLeft",
    "mouthLowerDownLeft",
    "mouthLowerDownRight",
    "mouthPressLeft",
    "mouthPressRight",
    "mouthPucker",
    "mouthRight",
    "mouthRollLower",
    "mouthRollUpper",
    "mouthShrugLower",
    "mouthShrugUpper",
    "mouthSmileLeft",
    "mouthSmileRight",
    "mouthStretchLeft",
    "mouthStretchRight",
    "mouthUpperUpLeft",
    "mouthUpperUpRight",
    "noseSneerLeft",
    "noseSneerRight",
    "tongueOut",
)


IMAGE_LEFT_BROW = (17, 18, 19, 20, 21)
IMAGE_RIGHT_BROW = (22, 23, 24, 25, 26)
IMAGE_LEFT_EYE = (36, 37, 38, 39, 40, 41)
IMAGE_RIGHT_EYE = (42, 43, 44, 45, 46, 47)
IMAGE_LEFT_MOUTH_CORNER = 48
IMAGE_RIGHT_MOUTH_CORNER = 54


def _clamp01(value: float) -> float:
    return float(np.clip(value, 0.0, 1.0))


def _safe_norm(vector: np.ndarray) -> float:
    return float(np.linalg.norm(vector))


def _distance(points: np.ndarray, a: int, b: int) -> float:
    return _safe_norm(points[a] - points[b])


def _mean_points(points: np.ndarray, indices: Sequence[int]) -> np.ndarray:
    return np.mean(points[np.asarray(indices, dtype=np.int32)], axis=0)


def _eye_openness(points: np.ndarray, eye_indices: Sequence[int]) -> float:
    a, b, c, d, e, f = eye_indices
    width = max(_distance(points, a, d), 1e-6)
    height = 0.5 * (_distance(points, b, f) + _distance(points, c, e))
    return height / width


def _to_numpy(vector: Sequence[float] | np.ndarray, *, name: str) -> np.ndarray:
    array = np.asarray(vector, dtype=np.float32).reshape(-1)
    if array.size == 0:
        raise ValueError(f"{name} must not be empty.")
    return array


@dataclass(frozen=True)
class Qualcomm3DMMCoefficients:
    identity: np.ndarray
    expression: np.ndarray
    pitch: float
    yaw: float
    roll: float
    translation_x: float
    translation_y: float
    focal_length: float

    @property
    def output_vector(self) -> np.ndarray:
        return np.concatenate(
            [
                self.identity.reshape(-1),
                self.expression.reshape(-1),
                np.asarray(
                    [
                        self.pitch,
                        self.yaw,
                        self.roll,
                        self.translation_x,
                        self.translation_y,
                        self.focal_length,
                    ],
                    dtype=np.float32,
                ),
            ]
        )


@dataclass(frozen=True)
class ReconstructedFace:
    landmarks_2d: np.ndarray
    landmarks_3d: np.ndarray
    pose_radians: Mapping[str, float]
    camera: Mapping[str, float]


def split_qualcomm_3dmm_output(
    output: Sequence[float] | np.ndarray,
) -> Qualcomm3DMMCoefficients:
    """
    Qualcomm FaceMap 3DMM output layout.

    Official post-processing uses:
    - identity coefficients: 219
    - expression coefficients: 39
    - pose/translation/focal: 6

    Total length: 264.
    """

    vector = _to_numpy(output, name="output")
    if vector.size != 264:
        raise ValueError(f"Qualcomm FaceMap output must have length 264, got {vector.size}.")

    return Qualcomm3DMMCoefficients(
        identity=vector[0:219].copy(),
        expression=vector[219:258].copy(),
        pitch=float(vector[258]),
        yaw=float(vector[259]),
        roll=float(vector[260]),
        translation_x=float(vector[261]),
        translation_y=float(vector[262]),
        focal_length=float(vector[263]),
    )


def transform_crop_landmarks_to_image(
    landmarks_2d: np.ndarray,
    bbox_xyxy: Sequence[float],
    resized_height: int,
    resized_width: int,
) -> np.ndarray:
    """
    Match Qualcomm's official coordinate restoration from crop space to image space.
    """

    points = np.asarray(landmarks_2d, dtype=np.float32).reshape(-1, 2).copy()
    x0, y0, x1, y1 = map(float, bbox_xyxy)
    width = max(x1 - x0, 1.0)
    height = max(y1 - y0, 1.0)

    points[:, 0] = (points[:, 0] + resized_width * 0.5) * width / resized_width + x0
    points[:, 1] = (points[:, 1] + resized_height * 0.5) * height / resized_height + y0
    return points


def reconstruct_qualcomm_68_landmarks(
    coefficients: Qualcomm3DMMCoefficients | Sequence[float] | np.ndarray,
    mean_face: np.ndarray,
    shape_basis: np.ndarray,
    blendshape_basis: np.ndarray,
) -> ReconstructedFace:
    """
    Reconstruct 68 3D landmarks following Qualcomm's published post-processing.

    Required asset shapes:
    - mean_face: (204,) or (68, 3)
    - shape_basis: (204, 219)
    - blendshape_basis: (204, 39)
    """

    coeffs = (
        coefficients
        if isinstance(coefficients, Qualcomm3DMMCoefficients)
        else split_qualcomm_3dmm_output(coefficients)
    )

    face = np.asarray(mean_face, dtype=np.float32).reshape(68 * 3, 1)
    basis_id = np.asarray(shape_basis, dtype=np.float32).reshape(68 * 3, 219)
    basis_exp = np.asarray(blendshape_basis, dtype=np.float32).reshape(68 * 3, 39)

    alpha_id = (coeffs.identity * 3.0).reshape(219, 1)
    alpha_exp = (coeffs.expression * 0.5 + 0.5).reshape(39, 1)
    pitch = float(coeffs.pitch) * np.pi / 2.0
    yaw = float(coeffs.yaw) * np.pi / 2.0
    roll = float(coeffs.roll) * np.pi / 2.0
    tx = float(coeffs.translation_x) * 60.0
    ty = float(coeffs.translation_y) * 60.0
    tz = 500.0
    focal = float(coeffs.focal_length) * 150.0 + 450.0

    p_matrix = np.asarray(
        [
            [1.0, 0.0, 0.0],
            [0.0, np.cos(-np.pi), -np.sin(-np.pi)],
            [0.0, np.sin(-np.pi), np.cos(-np.pi)],
        ],
        dtype=np.float32,
    )
    roll_matrix = np.asarray(
        [
            [np.cos(-roll), -np.sin(-roll), 0.0],
            [np.sin(-roll), np.cos(-roll), 0.0],
            [0.0, 0.0, 1.0],
        ],
        dtype=np.float32,
    )
    yaw_matrix = np.asarray(
        [
            [np.cos(-yaw), 0.0, np.sin(-yaw)],
            [0.0, 1.0, 0.0],
            [-np.sin(-yaw), 0.0, np.cos(-yaw)],
        ],
        dtype=np.float32,
    )
    pitch_matrix = np.asarray(
        [
            [1.0, 0.0, 0.0],
            [0.0, np.cos(-pitch), -np.sin(-pitch)],
            [0.0, np.sin(-pitch), np.cos(-pitch)],
        ],
        dtype=np.float32,
    )
    rotation = yaw_matrix @ pitch_matrix @ p_matrix @ roll_matrix

    vertices = (
        face + basis_id @ alpha_id + basis_exp @ alpha_exp
    ).reshape(68, 3) @ rotation.T
    vertices[:, 0] += tx
    vertices[:, 1] += ty
    vertices[:, 2] += tz

    projected = vertices[:, :2] * focal / tz
    return ReconstructedFace(
        landmarks_2d=projected.astype(np.float32),
        landmarks_3d=vertices.astype(np.float32),
        pose_radians={"pitch": pitch, "yaw": yaw, "roll": roll},
        camera={"focal_length": focal, "translation_x": tx, "translation_y": ty, "translation_z": tz},
    )


class Qualcomm3DMMToARKit52Converter:
    """
    Heuristic converter from Qualcomm 3DMM outputs to ARKit-style 52 blendshapes.

    This is not a learned semantic mapping because Qualcomm does not publish a direct
    39-exp -> ARKit52 conversion table. Instead, the converter:
    1. reconstructs 68 landmarks from the official 3DMM basis when assets are available
    2. extracts geometry proxies
    3. estimates the 52 ARKit blendshape values in [0, 1]

    Shapes that are impossible to infer reliably from 68 landmarks alone, such as
    `tongueOut`, are kept near zero.
    """

    def __init__(
        self,
        *,
        mirror_input: bool = False,
        neutral_momentum: float = 0.90,
        use_head_pose_as_eye_gaze: bool = False,
    ) -> None:
        self.mirror_input = mirror_input
        self.neutral_momentum = float(np.clip(neutral_momentum, 0.0, 0.999))
        self.use_head_pose_as_eye_gaze = use_head_pose_as_eye_gaze
        self._neutral_metrics: dict[str, float] | None = None
        self._neutral_expression: np.ndarray | None = None

    def reset(self) -> None:
        self._neutral_metrics = None
        self._neutral_expression = None

    def estimate_from_output(
        self,
        output: Sequence[float] | np.ndarray,
        *,
        mean_face: np.ndarray,
        shape_basis: np.ndarray,
        blendshape_basis: np.ndarray,
        update_neutral: bool = True,
    ) -> dict[str, float]:
        coeffs = split_qualcomm_3dmm_output(output)
        reconstructed = reconstruct_qualcomm_68_landmarks(
            coeffs,
            mean_face=mean_face,
            shape_basis=shape_basis,
            blendshape_basis=blendshape_basis,
        )
        return self.estimate_from_landmarks(
            reconstructed.landmarks_2d,
            expression_coeffs=coeffs.expression,
            pose_radians=reconstructed.pose_radians,
            update_neutral=update_neutral,
        )

    def estimate_from_landmarks(
        self,
        landmarks_2d: np.ndarray,
        *,
        expression_coeffs: Sequence[float] | np.ndarray | None = None,
        pose_radians: Mapping[str, float] | None = None,
        update_neutral: bool = True,
    ) -> dict[str, float]:
        points = np.asarray(landmarks_2d, dtype=np.float32).reshape(68, 2)
        metrics = self._extract_metrics(points)
        neutral = self._neutral_metrics or dict(metrics)
        shapes = self._metrics_to_blendshapes(metrics, neutral, pose_radians)

        if update_neutral and self._should_update_neutral(shapes, pose_radians):
            self._update_neutral(metrics, expression_coeffs)

        return {name: _clamp01(shapes.get(name, 0.0)) for name in ARKIT_52_BLENDSHAPES}

    def _resolve_sides(self) -> dict[str, object]:
        if self.mirror_input:
            left_eye = IMAGE_LEFT_EYE
            right_eye = IMAGE_RIGHT_EYE
            left_brow = IMAGE_LEFT_BROW
            right_brow = IMAGE_RIGHT_BROW
            left_corner = IMAGE_LEFT_MOUTH_CORNER
            right_corner = IMAGE_RIGHT_MOUTH_CORNER
            left_nose = 31
            right_nose = 35
            left_upper_outer = 49
            right_upper_outer = 53
            left_lower_outer = 59
            right_lower_outer = 55
        else:
            left_eye = IMAGE_RIGHT_EYE
            right_eye = IMAGE_LEFT_EYE
            left_brow = IMAGE_RIGHT_BROW
            right_brow = IMAGE_LEFT_BROW
            left_corner = IMAGE_RIGHT_MOUTH_CORNER
            right_corner = IMAGE_LEFT_MOUTH_CORNER
            left_nose = 35
            right_nose = 31
            left_upper_outer = 53
            right_upper_outer = 49
            left_lower_outer = 55
            right_lower_outer = 59

        return {
            "left_eye": left_eye,
            "right_eye": right_eye,
            "left_brow": left_brow,
            "right_brow": right_brow,
            "left_corner": left_corner,
            "right_corner": right_corner,
            "left_nose": left_nose,
            "right_nose": right_nose,
            "left_upper_outer": left_upper_outer,
            "right_upper_outer": right_upper_outer,
            "left_lower_outer": left_lower_outer,
            "right_lower_outer": right_lower_outer,
        }

    def _extract_metrics(self, points: np.ndarray) -> dict[str, float]:
        side = self._resolve_sides()
        face_width = max(_distance(points, 0, 16), 1e-6)
        face_height = max(abs(float(points[8, 1] - points[27, 1])), face_width * 0.6, 1e-6)
        mouth_center = 0.5 * (points[51] + points[57])
        mouth_inner_open = (
            _distance(points, 61, 67) + _distance(points, 62, 66) + _distance(points, 63, 65)
        ) / (3.0 * face_height)
        mouth_outer_open = (
            _distance(points, 50, 58) + _distance(points, 51, 57) + _distance(points, 52, 56)
        ) / (3.0 * face_height)
        mouth_width = _distance(points, 48, 54) / face_width
        left_eye_center = _mean_points(points, side["left_eye"])
        right_eye_center = _mean_points(points, side["right_eye"])
        left_brow = side["left_brow"]
        right_brow = side["right_brow"]
        left_corner = int(side["left_corner"])
        right_corner = int(side["right_corner"])

        metrics = {
            "face_width": face_width,
            "face_height": face_height,
            "left_eye_open": _eye_openness(points, side["left_eye"]),
            "right_eye_open": _eye_openness(points, side["right_eye"]),
            "mouth_open_inner": mouth_inner_open,
            "mouth_open_outer": mouth_outer_open,
            "mouth_width": mouth_width,
            "mouth_center_x": float(mouth_center[0]),
            "nose_x": float(points[33, 0]),
            "chin_x": float(points[8, 0]),
            "chin_drop": float(points[8, 1] - points[33, 1]) / face_height,
            "mouth_center_y": float(mouth_center[1]),
            "upper_lip_center_y": float(points[51, 1]),
            "lower_lip_center_y": float(points[57, 1]),
            "upper_lip_thickness": _distance(points, 51, 62) / face_height,
            "lower_lip_thickness": _distance(points, 57, 66) / face_height,
            "left_corner_raise": float(mouth_center[1] - points[left_corner, 1]) / face_height,
            "right_corner_raise": float(mouth_center[1] - points[right_corner, 1]) / face_height,
            "left_corner_stretch": float(abs(points[left_corner, 0] - mouth_center[0])) / face_width,
            "right_corner_stretch": float(abs(points[right_corner, 0] - mouth_center[0])) / face_width,
            "left_outer_brow_gap": float(left_eye_center[1] - points[left_brow[0], 1]) / face_height,
            "left_inner_brow_gap": float(left_eye_center[1] - points[left_brow[-1], 1]) / face_height,
            "right_inner_brow_gap": float(right_eye_center[1] - points[right_brow[0], 1]) / face_height,
            "right_outer_brow_gap": float(right_eye_center[1] - points[right_brow[-1], 1]) / face_height,
            "left_upper_nose_gap": float(points[int(side["left_upper_outer"]), 1] - points[int(side["left_nose"]), 1])
            / face_height,
            "right_upper_nose_gap": float(points[int(side["right_upper_outer"]), 1] - points[int(side["right_nose"]), 1])
            / face_height,
            "left_lower_chin_gap": float(points[8, 1] - points[int(side["left_lower_outer"]), 1]) / face_height,
            "right_lower_chin_gap": float(points[8, 1] - points[int(side["right_lower_outer"]), 1]) / face_height,
            "left_mouth_press_gap": _distance(points, int(side["left_upper_outer"]), 67 if self.mirror_input else 65)
            / face_height,
            "right_mouth_press_gap": _distance(points, int(side["right_upper_outer"]), 65 if self.mirror_input else 67)
            / face_height,
        }
        metrics["inner_brow_gap"] = 0.5 * (
            metrics["left_inner_brow_gap"] + metrics["right_inner_brow_gap"]
        )
        return metrics

    def _delta(self, metrics: Mapping[str, float], neutral: Mapping[str, float], key: str, scale: float) -> float:
        return (float(metrics[key]) - float(neutral[key])) / max(scale, 1e-6)

    def _metrics_to_blendshapes(
        self,
        metrics: Mapping[str, float],
        neutral: Mapping[str, float],
        pose_radians: Mapping[str, float] | None,
    ) -> dict[str, float]:
        face_height = float(metrics["face_height"])
        mouth_open_abs = _clamp01((float(metrics["mouth_open_inner"]) - 0.010) / 0.070)
        mouth_width_abs = _clamp01((float(metrics["mouth_width"]) - 0.34) / 0.22)
        left_blink_abs = _clamp01((0.30 - float(metrics["left_eye_open"])) / 0.18)
        right_blink_abs = _clamp01((0.30 - float(metrics["right_eye_open"])) / 0.18)
        left_wide_abs = _clamp01((float(metrics["left_eye_open"]) - 0.32) / 0.10)
        right_wide_abs = _clamp01((float(metrics["right_eye_open"]) - 0.32) / 0.10)

        left_blink = max(
            left_blink_abs,
            _clamp01(-self._delta(metrics, neutral, "left_eye_open", 0.08)),
        )
        right_blink = max(
            right_blink_abs,
            _clamp01(-self._delta(metrics, neutral, "right_eye_open", 0.08)),
        )
        jaw_open = max(
            mouth_open_abs,
            _clamp01(self._delta(metrics, neutral, "mouth_open_inner", 0.060)),
            _clamp01(self._delta(metrics, neutral, "chin_drop", 0.18)),
        )
        mouth_smile_left = max(
            _clamp01((float(metrics["left_corner_raise"]) - 0.010) / 0.080),
            _clamp01(self._delta(metrics, neutral, "left_corner_raise", 0.050)),
        )
        mouth_smile_right = max(
            _clamp01((float(metrics["right_corner_raise"]) - 0.010) / 0.080),
            _clamp01(self._delta(metrics, neutral, "right_corner_raise", 0.050)),
        )
        mouth_frown_left = max(
            _clamp01((-float(metrics["left_corner_raise"]) - 0.005) / 0.080),
            _clamp01(-self._delta(metrics, neutral, "left_corner_raise", 0.050)),
        )
        mouth_frown_right = max(
            _clamp01((-float(metrics["right_corner_raise"]) - 0.005) / 0.080),
            _clamp01(-self._delta(metrics, neutral, "right_corner_raise", 0.050)),
        )
        left_outer_up = max(
            _clamp01((float(metrics["left_outer_brow_gap"]) - 0.09) / 0.08),
            _clamp01(self._delta(metrics, neutral, "left_outer_brow_gap", 0.05)),
        )
        right_outer_up = max(
            _clamp01((float(metrics["right_outer_brow_gap"]) - 0.09) / 0.08),
            _clamp01(self._delta(metrics, neutral, "right_outer_brow_gap", 0.05)),
        )
        brow_inner_up = max(
            _clamp01((float(metrics["inner_brow_gap"]) - 0.095) / 0.08),
            _clamp01(self._delta(metrics, neutral, "inner_brow_gap", 0.05)),
        )
        brow_down_left = max(
            _clamp01((0.085 - float(metrics["left_inner_brow_gap"])) / 0.060),
            _clamp01(-self._delta(metrics, neutral, "left_inner_brow_gap", 0.04)),
        ) * (1.0 - 0.35 * left_blink)
        brow_down_right = max(
            _clamp01((0.085 - float(metrics["right_inner_brow_gap"])) / 0.060),
            _clamp01(-self._delta(metrics, neutral, "right_inner_brow_gap", 0.04)),
        ) * (1.0 - 0.35 * right_blink)

        mouth_pucker = max(
            _clamp01((0.43 - float(metrics["mouth_width"])) / 0.18) * _clamp01((0.050 - jaw_open) / 0.050),
            _clamp01(-self._delta(metrics, neutral, "mouth_width", 0.12)),
        )
        mouth_funnel = (
            _clamp01((0.48 - float(metrics["mouth_width"])) / 0.20)
            * _clamp01((float(metrics["mouth_open_outer"]) - 0.02) / 0.07)
        )
        mouth_stretch_left = max(
            _clamp01((float(metrics["left_corner_stretch"]) - 0.17) / 0.10),
            _clamp01(self._delta(metrics, neutral, "left_corner_stretch", 0.06)),
        )
        mouth_stretch_right = max(
            _clamp01((float(metrics["right_corner_stretch"]) - 0.17) / 0.10),
            _clamp01(self._delta(metrics, neutral, "right_corner_stretch", 0.06)),
        )
        mouth_center_shift = float(metrics["mouth_center_x"] - metrics["nose_x"]) / max(float(metrics["face_width"]), 1e-6)
        jaw_shift = mouth_center_shift + 0.55 * (
            float(metrics["chin_x"] - metrics["nose_x"]) / max(float(metrics["face_width"]), 1e-6)
        )

        if pose_radians:
            jaw_shift -= 0.12 * float(pose_radians.get("yaw", 0.0))

        upper_up_left = max(
            _clamp01((0.19 - float(metrics["left_upper_nose_gap"])) / 0.10),
            _clamp01(-self._delta(metrics, neutral, "left_upper_nose_gap", 0.05)),
        )
        upper_up_right = max(
            _clamp01((0.19 - float(metrics["right_upper_nose_gap"])) / 0.10),
            _clamp01(-self._delta(metrics, neutral, "right_upper_nose_gap", 0.05)),
        )
        lower_down_left = max(
            jaw_open * 0.55 + mouth_frown_left * 0.25,
            _clamp01(-self._delta(metrics, neutral, "left_lower_chin_gap", 0.07)),
        )
        lower_down_right = max(
            jaw_open * 0.55 + mouth_frown_right * 0.25,
            _clamp01(-self._delta(metrics, neutral, "right_lower_chin_gap", 0.07)),
        )
        mouth_press_left = max(
            _clamp01((0.045 - float(metrics["left_mouth_press_gap"])) / 0.03) * _clamp01((0.030 - jaw_open) / 0.030),
            _clamp01(-self._delta(metrics, neutral, "left_mouth_press_gap", 0.02)),
        )
        mouth_press_right = max(
            _clamp01((0.045 - float(metrics["right_mouth_press_gap"])) / 0.03) * _clamp01((0.030 - jaw_open) / 0.030),
            _clamp01(-self._delta(metrics, neutral, "right_mouth_press_gap", 0.02)),
        )

        chin_drop_delta = _clamp01(self._delta(metrics, neutral, "chin_drop", 0.12))
        mouth_close = _clamp01((chin_drop_delta - jaw_open) * 1.6)
        mouth_roll_upper = max(
            _clamp01((0.040 - float(metrics["upper_lip_thickness"])) / 0.025),
            _clamp01(-self._delta(metrics, neutral, "upper_lip_thickness", 0.018)),
        ) * _clamp01((0.030 - float(metrics["mouth_open_inner"])) / 0.030)
        mouth_roll_lower = max(
            _clamp01((0.040 - float(metrics["lower_lip_thickness"])) / 0.025),
            _clamp01(-self._delta(metrics, neutral, "lower_lip_thickness", 0.018)),
        ) * _clamp01((0.030 - float(metrics["mouth_open_inner"])) / 0.030)
        mouth_shrug_upper = _clamp01(upper_up_left * 0.5 + upper_up_right * 0.5 + mouth_close * 0.2)
        mouth_shrug_lower = _clamp01(
            max(
                _clamp01((float(metrics["left_lower_chin_gap"]) - 0.24) / 0.12),
                _clamp01((float(metrics["right_lower_chin_gap"]) - 0.24) / 0.12),
            )
            * 0.7
            + mouth_close * 0.2
        )
        cheek_puff = _clamp01(mouth_pucker * 0.7 * (1.0 - jaw_open))
        cheek_squint_left = _clamp01(0.5 * mouth_smile_left + 0.35 * (1.0 - left_wide_abs) + 0.2 * left_blink)
        cheek_squint_right = _clamp01(
            0.5 * mouth_smile_right + 0.35 * (1.0 - right_wide_abs) + 0.2 * right_blink
        )
        nose_sneer_left = _clamp01(0.55 * upper_up_left + 0.25 * mouth_smile_left + 0.15 * cheek_squint_left)
        nose_sneer_right = _clamp01(0.55 * upper_up_right + 0.25 * mouth_smile_right + 0.15 * cheek_squint_right)

        shapes = {name: 0.0 for name in ARKIT_52_BLENDSHAPES}
        shapes["browDownLeft"] = brow_down_left
        shapes["browDownRight"] = brow_down_right
        shapes["browInnerUp"] = brow_inner_up
        shapes["browOuterUpLeft"] = left_outer_up
        shapes["browOuterUpRight"] = right_outer_up
        shapes["cheekPuff"] = cheek_puff
        shapes["cheekSquintLeft"] = cheek_squint_left
        shapes["cheekSquintRight"] = cheek_squint_right
        shapes["eyeBlinkLeft"] = left_blink
        shapes["eyeBlinkRight"] = right_blink
        shapes["eyeSquintLeft"] = _clamp01(left_blink * 0.65 + cheek_squint_left * 0.25)
        shapes["eyeSquintRight"] = _clamp01(right_blink * 0.65 + cheek_squint_right * 0.25)
        shapes["eyeWideLeft"] = max(left_wide_abs, _clamp01(self._delta(metrics, neutral, "left_eye_open", 0.08)))
        shapes["eyeWideRight"] = max(
            right_wide_abs,
            _clamp01(self._delta(metrics, neutral, "right_eye_open", 0.08)),
        )
        shapes["jawForward"] = _clamp01(max(mouth_pucker, mouth_funnel) * 0.25)
        shapes["jawLeft"] = _clamp01(max(-jaw_shift, 0.0) / 0.06)
        shapes["jawOpen"] = jaw_open
        shapes["jawRight"] = _clamp01(max(jaw_shift, 0.0) / 0.06)
        shapes["mouthClose"] = mouth_close
        shapes["mouthDimpleLeft"] = _clamp01(mouth_smile_left * 0.55 + mouth_stretch_left * 0.25)
        shapes["mouthDimpleRight"] = _clamp01(mouth_smile_right * 0.55 + mouth_stretch_right * 0.25)
        shapes["mouthFrownLeft"] = mouth_frown_left
        shapes["mouthFrownRight"] = mouth_frown_right
        shapes["mouthFunnel"] = mouth_funnel
        shapes["mouthLeft"] = _clamp01(max(-mouth_center_shift, 0.0) / 0.05)
        shapes["mouthLowerDownLeft"] = _clamp01(lower_down_left)
        shapes["mouthLowerDownRight"] = _clamp01(lower_down_right)
        shapes["mouthPressLeft"] = mouth_press_left
        shapes["mouthPressRight"] = mouth_press_right
        shapes["mouthPucker"] = mouth_pucker
        shapes["mouthRight"] = _clamp01(max(mouth_center_shift, 0.0) / 0.05)
        shapes["mouthRollLower"] = mouth_roll_lower
        shapes["mouthRollUpper"] = mouth_roll_upper
        shapes["mouthShrugLower"] = mouth_shrug_lower
        shapes["mouthShrugUpper"] = mouth_shrug_upper
        shapes["mouthSmileLeft"] = mouth_smile_left
        shapes["mouthSmileRight"] = mouth_smile_right
        shapes["mouthStretchLeft"] = max(mouth_stretch_left, mouth_width_abs * 0.55)
        shapes["mouthStretchRight"] = max(mouth_stretch_right, mouth_width_abs * 0.55)
        shapes["mouthUpperUpLeft"] = upper_up_left
        shapes["mouthUpperUpRight"] = upper_up_right
        shapes["noseSneerLeft"] = nose_sneer_left
        shapes["noseSneerRight"] = nose_sneer_right

        if self.use_head_pose_as_eye_gaze and pose_radians:
            yaw = float(pose_radians.get("yaw", 0.0))
            pitch = float(pose_radians.get("pitch", 0.0))
            left_amount = _clamp01(max(-yaw, 0.0) / 0.35)
            right_amount = _clamp01(max(yaw, 0.0) / 0.35)
            up_amount = _clamp01(max(-pitch, 0.0) / 0.25)
            down_amount = _clamp01(max(pitch, 0.0) / 0.25)
            shapes["eyeLookInLeft"] = left_amount
            shapes["eyeLookOutLeft"] = right_amount
            shapes["eyeLookInRight"] = right_amount
            shapes["eyeLookOutRight"] = left_amount
            shapes["eyeLookUpLeft"] = up_amount
            shapes["eyeLookUpRight"] = up_amount
            shapes["eyeLookDownLeft"] = down_amount
            shapes["eyeLookDownRight"] = down_amount

        return shapes

    def _should_update_neutral(
        self,
        shapes: Mapping[str, float],
        pose_radians: Mapping[str, float] | None,
    ) -> bool:
        yaw = abs(float((pose_radians or {}).get("yaw", 0.0)))
        pitch = abs(float((pose_radians or {}).get("pitch", 0.0)))
        roll = abs(float((pose_radians or {}).get("roll", 0.0)))
        if yaw > 0.45 or pitch > 0.35 or roll > 0.45:
            return False

        active = max(
            float(shapes["jawOpen"]),
            float(shapes["mouthSmileLeft"]),
            float(shapes["mouthSmileRight"]),
            float(shapes["mouthFrownLeft"]),
            float(shapes["mouthFrownRight"]),
            float(shapes["eyeBlinkLeft"]),
            float(shapes["eyeBlinkRight"]),
            float(shapes["browInnerUp"]),
            float(shapes["browDownLeft"]),
            float(shapes["browDownRight"]),
        )
        return active < 0.28

    def _update_neutral(
        self,
        metrics: Mapping[str, float],
        expression_coeffs: Sequence[float] | np.ndarray | None,
    ) -> None:
        if self._neutral_metrics is None:
            self._neutral_metrics = {key: float(value) for key, value in metrics.items()}
        else:
            keep = self.neutral_momentum
            self._neutral_metrics = {
                key: keep * float(self._neutral_metrics[key]) + (1.0 - keep) * float(value)
                for key, value in metrics.items()
            }

        if expression_coeffs is None:
            return

        expr = _to_numpy(expression_coeffs, name="expression_coeffs")
        if self._neutral_expression is None:
            self._neutral_expression = expr.copy()
            return

        keep = self.neutral_momentum
        self._neutral_expression = keep * self._neutral_expression + (1.0 - keep) * expr


def estimate_arkit52_from_qualcomm_output(
    output: Sequence[float] | np.ndarray,
    *,
    mean_face: np.ndarray,
    shape_basis: np.ndarray,
    blendshape_basis: np.ndarray,
    mirror_input: bool = False,
    use_head_pose_as_eye_gaze: bool = False,
) -> dict[str, float]:
    converter = Qualcomm3DMMToARKit52Converter(
        mirror_input=mirror_input,
        use_head_pose_as_eye_gaze=use_head_pose_as_eye_gaze,
    )
    return converter.estimate_from_output(
        output,
        mean_face=mean_face,
        shape_basis=shape_basis,
        blendshape_basis=blendshape_basis,
    )


__all__ = [
    "ARKIT_52_BLENDSHAPES",
    "Qualcomm3DMMCoefficients",
    "Qualcomm3DMMToARKit52Converter",
    "ReconstructedFace",
    "estimate_arkit52_from_qualcomm_output",
    "reconstruct_qualcomm_68_landmarks",
    "split_qualcomm_3dmm_output",
    "transform_crop_landmarks_to_image",
]



def estimate_arkit52_from_landmarks_official_plus_heuristic(landmarks_crop_xy, expression_coeffs, pose_radians):
    # [HEURISTIC] There is no official Qualcomm facemap_3dmm -> ARKit52 mapping table.
    # We therefore estimate ARKit-style controls from reconstructed 68 landmarks and head pose.
    converter = Qualcomm3DMMToARKit52Converter(
        mirror_input=False,
        neutral_momentum=0.90,
        use_head_pose_as_eye_gaze=True,
    )
    return converter.estimate_from_landmarks(
        landmarks_crop_xy,
        expression_coeffs=expression_coeffs,
        pose_radians=pose_radians,
        update_neutral=False,
    )


def run_single_image_pipeline(image_bgr, *, face_app, interpreter, mean_face, shape_basis, blendshape_basis, pad_ratio=0.30):
    faces = face_app.get(image_bgr)
    if not faces:
        raise RuntimeError("InsightFace did not detect any face.")

    largest_face = get_largest_face(faces)
    det_bbox_xyxy = np.asarray(largest_face.bbox[:4], dtype=np.float32)
    crop_bbox_xyxy = make_square_crop_bbox(image_bgr.shape, det_bbox_xyxy, pad_ratio=pad_ratio)
    _, crop_bgr_128 = crop_and_resize_face(image_bgr, crop_bbox_xyxy, output_size=MODEL_INPUT_SIZE)

    raw_output = run_facemap_tflite(interpreter, crop_bgr_128)
    coeff_info = split_facemap_output(raw_output)

    # [OFFICIAL] 68 landmark reconstruction using meanFace / shapeBasis / blendShape
    reconstructed = reconstruct_qualcomm_68_landmarks(
        coeff_info["used_output"],
        mean_face=mean_face,
        shape_basis=shape_basis,
        blendshape_basis=blendshape_basis,
    )

    landmarks_crop_xy = reconstructed.landmarks_2d.copy()
    landmarks_image_xy = transform_crop_landmarks_to_image(
        landmarks_crop_xy,
        crop_bbox_xyxy,
        resized_height=MODEL_INPUT_SIZE,
        resized_width=MODEL_INPUT_SIZE,
    )

    blendshapes_52 = estimate_arkit52_from_landmarks_official_plus_heuristic(
        landmarks_crop_xy,
        expression_coeffs=coeff_info["expression"],
        pose_radians=reconstructed.pose_radians,
    )

    detection_vis_bgr = draw_detection_overlay(image_bgr, det_bbox_xyxy, crop_bbox_xyxy)
    crop_landmark_vis_bgr = draw_landmarks_on_crop(crop_bgr_128, landmarks_crop_xy)
    image_landmark_vis_bgr = draw_landmarks_on_image(detection_vis_bgr, landmarks_image_xy)
    chart_fig = render_blendshape_bar_chart(blendshapes_52)
    blendshape_vis_bgr = figure_to_bgr(chart_fig)

    result = {
        "image_shape_hw": [int(image_bgr.shape[0]), int(image_bgr.shape[1])],
        "det_bbox_xyxy": det_bbox_xyxy.astype(np.float32),
        "crop_bbox_xyxy": crop_bbox_xyxy.astype(np.float32),
        "crop_pad_ratio": float(pad_ratio),
        "crop_bgr_128": crop_bgr_128,
        "coeff_info": coeff_info,
        "reconstructed": reconstructed,
        "landmarks_crop_xy": landmarks_crop_xy.astype(np.float32),
        "landmarks_image_xy": landmarks_image_xy.astype(np.float32),
        "blendshapes_52": blendshapes_52,
        "original_rgb": cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB),
        "detection_vis_rgb": cv2.cvtColor(detection_vis_bgr, cv2.COLOR_BGR2RGB),
        "crop_rgb": cv2.cvtColor(crop_bgr_128, cv2.COLOR_BGR2RGB),
        "crop_landmark_vis_rgb": cv2.cvtColor(crop_landmark_vis_bgr, cv2.COLOR_BGR2RGB),
        "image_landmark_vis_rgb": cv2.cvtColor(image_landmark_vis_bgr, cv2.COLOR_BGR2RGB),
        "blendshape_vis_bgr": blendshape_vis_bgr,
        "blendshape_vis_rgb": cv2.cvtColor(blendshape_vis_bgr, cv2.COLOR_BGR2RGB),
    }
    return result


download_first_available(TFLITE_URLS, TFLITE_MODEL_PATH)
interpreter = build_tflite_interpreter(TFLITE_MODEL_PATH)
mean_face = load_numpy(CachedWebModelAsset.from_asset_store(MODEL_ID, MODEL_ASSET_VERSION, "meanFace.npy"))
shape_basis = load_numpy(CachedWebModelAsset.from_asset_store(MODEL_ID, MODEL_ASSET_VERSION, "shapeBasis.npy"))
blendshape_basis = load_numpy(CachedWebModelAsset.from_asset_store(MODEL_ID, MODEL_ASSET_VERSION, "blendShape.npy"))

face_app = FaceAnalysis(
    name="buffalo_l",
    allowed_modules=["detection"],
    providers=["CPUExecutionProvider"],
)
face_app.prepare(ctx_id=-1, det_size=(640, 640))

print("TFLite model:", TFLITE_MODEL_PATH)
print("Input details:", interpreter.get_input_details()[0]["shape"], interpreter.get_input_details()[0]["dtype"])
print("Output details:", interpreter.get_output_details()[0]["shape"], interpreter.get_output_details()[0]["dtype"])
print("Basis assets:", mean_face.shape, shape_basis.shape, blendshape_basis.shape)

ModuleNotFoundError: No module named 'numpy.char'

In [ ]:
# Upload image and run the full single-image pipeline in memory.
file_name, image_bgr = upload_one_image_bgr()
result = run_single_image_pipeline(
    image_bgr,
    face_app=face_app,
    interpreter=interpreter,
    mean_face=mean_face,
    shape_basis=shape_basis,
    blendshape_basis=blendshape_basis,
    pad_ratio=0.30,  # Assumption: use a slightly expanded square crop for 3DMM stability.
)

print("uploaded file:", file_name)
print("image shape:", result["image_shape_hw"])
print("InsightFace det bbox xyxy:", result["det_bbox_xyxy"].tolist())
print("facemap crop bbox xyxy:", result["crop_bbox_xyxy"].tolist())

show_rgb(result["original_rgb"], "Original image", figsize=(7, 7))
show_rgb(result["detection_vis_rgb"], "InsightFace bbox + facemap crop bbox", figsize=(7, 7))
show_rgb(result["crop_rgb"], "facemap_3dmm input crop (128x128)", figsize=(5, 5))

In [ ]:
# coeff inspection
coeff_info = result["coeff_info"]
print("Actual raw output dimension from TFLite:", coeff_info["raw_dim"])
print("Public 68-landmark decoder uses dimension:", coeff_info["used_dim"])
print("Unused tail dimension:", coeff_info["unused_tail"].size)
if coeff_info["unused_tail"].size > 0:
    print("Unused tail values:", coeff_info["unused_tail"].tolist())

summary_df = pd.DataFrame(
    [
        ["identity", 0, 219, 219, "used by public decoder"],
        ["expression", 219, 258, 39, "used by public decoder"],
        ["pose", 258, 261, 3, "pitch, yaw, roll"],
        ["translation/focal", 261, 264, 3, "tx, ty, focal"],
        ["unused tail", 264, int(coeff_info["raw_dim"]), int(coeff_info["unused_tail"].size), "not used by public 68-point decoder"],
    ],
    columns=["section", "start", "end_exclusive", "length", "note"],
)
display(summary_df)

print("[identity coeff] length =", coeff_info["identity"].size)
display(pd.DataFrame({"identity_coeff": coeff_info["identity"]}))

print("[expression coeff] length =", coeff_info["expression"].size)
display(pd.DataFrame({"expression_coeff": coeff_info["expression"]}))

pose_df = pd.DataFrame(
    {
        "name": ["pitch", "yaw", "roll"],
        "value": coeff_info["pose"],
    }
)
tf_df = pd.DataFrame(
    {
        "name": ["translation_x", "translation_y", "focal_length"],
        "value": coeff_info["translation_focal"],
    }
)
print("[pose related values]")
display(pose_df)
print("[translation / focal related values]")
display(tf_df)

print("[raw coeff vector head/tail preview]")
print("head(20):", np.round(coeff_info["raw_output"][:20], 6).tolist())
print("tail(20):", np.round(coeff_info["raw_output"][-20:], 6).tolist())

In [ ]:
# 68 landmark reconstruction and visualization
print("[OFFICIAL] landmarks were reconstructed with meanFace / shapeBasis / blendShape")
print("crop landmarks shape:", result["landmarks_crop_xy"].shape)
print("image landmarks shape:", result["landmarks_image_xy"].shape)

landmark_preview_df = pd.DataFrame(
    {
        "index": np.arange(68, dtype=np.int32),
        "crop_x": result["landmarks_crop_xy"][:, 0],
        "crop_y": result["landmarks_crop_xy"][:, 1],
        "image_x": result["landmarks_image_xy"][:, 0],
        "image_y": result["landmarks_image_xy"][:, 1],
    }
)
display(landmark_preview_df)

show_rgb(result["crop_landmark_vis_rgb"], "68 landmarks in crop coordinates", figsize=(6, 6))
show_rgb(result["image_landmark_vis_rgb"], "68 landmarks on original image with indices", figsize=(9, 9))

In [ ]:
# 52 blendshape conversion
print("[HEURISTIC] Qualcomm does not publish an official facemap_3dmm -> ARKit52 mapping.")
print("[HEURISTIC] The values below are estimated from reconstructed 68 landmarks + pose.")

blendshape_df = pd.DataFrame(
    [{"name": key, "value": float(value)} for key, value in result["blendshapes_52"].items()]
).sort_values("value", ascending=False, ignore_index=True)
display(blendshape_df)

required_keys = [
    "jawOpen",
    "eyeBlinkLeft",
    "eyeBlinkRight",
    "mouthSmileLeft",
    "mouthSmileRight",
    "browInnerUp",
]
print("Required blendshapes:")
for key in required_keys:
    print(f"  {key}: {result['blendshapes_52'][key]:.6f}")

show_rgb(result["blendshape_vis_rgb"], "Blendshape bar chart", figsize=(10, 14))

In [ ]:
# Final save/download cell only
OUTPUT_DIR = Path("/content/facemap_single_image_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

coeff_json = {
    "raw_dim": int(result["coeff_info"]["raw_dim"]),
    "used_dim_for_public_decoder": int(result["coeff_info"]["used_dim"]),
    "raw_output": result["coeff_info"]["raw_output"].astype(float).tolist(),
    "identity_coeff": result["coeff_info"]["identity"].astype(float).tolist(),
    "expression_coeff": result["coeff_info"]["expression"].astype(float).tolist(),
    "pose_values": {
        "pitch": float(result["coeff_info"]["pose"][0]),
        "yaw": float(result["coeff_info"]["pose"][1]),
        "roll": float(result["coeff_info"]["pose"][2]),
    },
    "translation_focal_values": {
        "translation_x": float(result["coeff_info"]["translation_focal"][0]),
        "translation_y": float(result["coeff_info"]["translation_focal"][1]),
        "focal_length": float(result["coeff_info"]["translation_focal"][2]),
    },
    "unused_tail": result["coeff_info"]["unused_tail"].astype(float).tolist(),
    "decoder_note": "Public 68-point decoder uses first 264 dims = 219 identity + 39 expression + 6 pose/translation/focal.",
}

landmarks_json = {
    "det_bbox_xyxy": result["det_bbox_xyxy"].astype(float).tolist(),
    "crop_bbox_xyxy": result["crop_bbox_xyxy"].astype(float).tolist(),
    "crop_pad_ratio": float(result["crop_pad_ratio"]),
    "crop_size": MODEL_INPUT_SIZE,
    "landmarks_crop_xy": result["landmarks_crop_xy"].astype(float).tolist(),
    "landmarks_image_xy": result["landmarks_image_xy"].astype(float).tolist(),
}

blendshape_json = {
    "schema": "arkit52_single_frame",
    "source": "qualcomm_facemap_3dmm_tflite_w8a8",
    "consumer_hint": "Use frames[0].blendshapes with ARKit/MediaPipe-style morph target names in three.js or Google raccoon GLB demos.",
    "official_steps": {
        "bbox_detector": "InsightFace FaceAnalysis",
        "landmark_decoder": "Qualcomm public 68-point 3DMM decoder with meanFace/shapeBasis/blendShape",
    },
    "heuristic_note": "ARKit52 values are estimated from reconstructed 68 landmarks and pose because Qualcomm does not publish an official direct mapping.",
    "fps": 30,
    "frames": [
        {
            "frame_index": 0,
            "blendshapes": {k: float(v) for k, v in result["blendshapes_52"].items()},
        }
    ],
}

coeff_path = OUTPUT_DIR / "coeff.json"
landmarks_path = OUTPUT_DIR / "landmarks68.json"
blendshape_path = OUTPUT_DIR / "blendshape_52.json"
landmark_vis_path = OUTPUT_DIR / "landmark_vis.png"
blendshape_vis_path = OUTPUT_DIR / "blendshape_vis.png"

coeff_path.write_text(json.dumps(coeff_json, ensure_ascii=False, indent=2))
landmarks_path.write_text(json.dumps(landmarks_json, ensure_ascii=False, indent=2))
blendshape_path.write_text(json.dumps(blendshape_json, ensure_ascii=False, indent=2))

cv2.imwrite(str(landmark_vis_path), cv2.cvtColor(result["image_landmark_vis_rgb"], cv2.COLOR_RGB2BGR))
cv2.imwrite(str(blendshape_vis_path), result["blendshape_vis_bgr"])

print("Saved files:")
print(coeff_path)
print(landmarks_path)
print(blendshape_path)
print(landmark_vis_path)
print(blendshape_vis_path)

files.download(str(coeff_path))
files.download(str(landmarks_path))
files.download(str(blendshape_path))
files.download(str(landmark_vis_path))
files.download(str(blendshape_vis_path))